**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kauvar-2025-001347"], check=True)
    os.chdir("/content/encore/entries/kauvar-2025-001347")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Kauvar, Richman, Liu et al. (2025): human eyepuff behaviour and dissociation, from DANDI:001347

Paper: Kauvar, Richman, Liu et al., "Conserved brain-wide emergence of emotional response from sensory
experience in humans and mice", Science 388, eadt3971 (2025), doi:10.1126/science.adt3971.

Dataset: [DANDI:001347](https://dandiarchive.org/dandiset/001347/0.250528.0702), version
0.250528.0702. It is the only published version (28 May 2025, the day before the paper appeared); the
paper cites the dandiset without a version, so the latest published version is pinned. Its content
(207 assets, 16.0 GB) is the same size and asset count as the current draft.

Claims file: `claims/001347/10.1126-science.adt3971.yaml`, written from the version of record with
its Materials and Methods, figure legends and the Supplementary Materials (Table S3).

This notebook tests the claims on human eye closure and dissociation (CADSS). Other notebooks of the
entry test the mouse behaviour (02), the iEEG spectrograms (03), the iEEG coding dimension (04) and the
mouse Neuropixels data of DANDI:001326 (05).

| Claim | Tested here |
|---|---|
| p50-condition-labels-link (link) | tested |
| cadss-scores-link (link) | tested |
| fig1e-human-late-eye-closure-ketamine | tested (with the code's condition labels) |
| figs1b-human-early-eye-closure-unchanged | tested (with the code's condition labels) |
| figs1c-human-late-eye-closure-decrease | tested (with the code's condition labels) |
| figs1d-human-cadss-saline-ketamine | tested (CADSS from the code record) |
| fig1f-human-subjective-reports | blocked: the verbal reports are not in the dataset |
| figs8a-ieeg-inclusion-per-participant | tested |
| figs8b-ieeg-excluded-participants | tested |
| fig2c-ieeg-cohort-selective-affective-decrease | tested |
| fig2d-ieeg-cadss-reversible | tested (CADSS from the code record) |
| figs14g-cadss-correlates-with-affective-decrease | tested (CADSS from the code record) |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

import dandi_io
import human_eye as he

print("dandiset", dandi_io.DANDISET, "version", dandi_io.VERSION)
assets = dandi_io.list_assets()
print(len(assets), "assets in the pinned version")

## Data check

The non-iEEG files (`sub-h*`) hold two series of 40 eye-closure trials each, `pre_trials_data` and
`post_trials_data`, with a time axis `time_reference/tt` (seconds from the puff). The iEEG behaviour
files (`sub-SD*_ses-puff_behavior.nwb`) hold, per session, the continuous eye closure, the video frame
of each puff (`events-*-on`), the trial windows (`by_trial_closure`, 800 samples by trials), the
stored per-trial early, late and late/early values, the early and late windows, the frame rate, and
`which_trials`.

In [ ]:
p50 = {}
rows = []
for sub in he.P50_SUBJECTS:
    tt, pre, post = he.load_p50(sub)
    p50[sub] = (tt, pre, post)
    for name, x in [("pre", pre), ("post", post)]:
        rows.append({"subject": sub, "series": name, "trials": x.shape[0], "samples": x.shape[1],
                     "t first (s)": tt[0], "t last (s)": round(tt[-1], 3),
                     "rate (Hz)": round(1 / np.median(np.diff(tt)), 3),
                     "min": round(np.nanmin(x), 3), "max": round(np.nanmax(x), 3),
                     "NaN": int(np.isnan(x).sum())})
print("Non-iEEG cohort (eye closure, fraction of the open eye):")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
beh = {sub: he.load_ieeg_behavior(sub) for sub in he.IEEG_SUBJECTS}
rows = []
for sub, b in beh.items():
    for sess in he.SESSIONS:
        s = b[sess]
        rows.append({"subject": sub, "session": sess, "file session": s["file_session"],
                     "trials": s["trials"].shape[0], "samples": s["trials"].shape[1],
                     "puffs": len(s["events_on"]), "which_trials": len(b["which_trials"]),
                     "continuous (min)": round(len(s["eyeclosure"]) / b["fps"] / 60, 1),
                     "trace min": round(np.nanmin(s["trials"]), 2),
                     "trace max": round(np.nanmax(s["trials"]), 2)})
info = pd.DataFrame(rows)
print("iEEG cohort behaviour files:")
print(info.to_string(index=False))
print("windows stored in every file: early", {tuple(b["ref_w"]) for b in beh.values()},
      "late", {tuple(b["aff_w"]) for b in beh.values()}, "fps", {b["fps"] for b in beh.values()})

Checks of the stored values against the traces they come from. (1) Each trial window of
`by_trial_closure` should be the continuous trace from 400 samples before to 400 after the puff
frame. (2) The stored early and late values should be the window means of the trial windows, and the
stored late/early their ratio. (3) The puff frames should be spaced as the Methods describe: blocks of
8 puffs 3 to 8 s apart, 35 s between blocks.

In [ ]:
rows = []
for sub, b in beh.items():
    for sess in he.SESSIONS:
        s = b[sess]
        ec, ev, tr = s["eyeclosure"], s["events_on"], s["trials"]
        ok = [i for i, e in enumerate(ev) if e - 400 >= 0 and e + 400 <= len(ec)]
        d_win = max(np.nanmax(np.abs(ec[ev[i] - 400:ev[i] + 400] - tr[i])) for i in ok)
        early = he.window_mean(tr, he.EARLY_W)
        late = he.window_mean(tr, he.LATE_W)
        gaps = np.diff(ev) / b["fps"]
        rows.append({"subject": sub, "session": sess, "windows checked": len(ok),
                     "max |window - trace|": d_win,
                     "max |early - stored|": np.nanmax(np.abs(early - s["reflexive"])),
                     "max |late - stored|": np.nanmax(np.abs(late - s["affective"])),
                     "max |ratio - stored|": np.nanmax(np.abs(late / early - s["affective_normed"])),
                     "puff gap min (s)": round(gaps.min(), 2), "median": round(np.median(gaps), 2),
                     "max": round(gaps.max(), 2), "gaps > 20 s": int((gaps > 20).sum())})
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))

The trial windows equal the continuous trace around each puff frame, and the stored per-trial values
equal the window means and their ratio to rounding, in every participant and session, so the stored
values and the traces agree (the table above). The shortest gap between puffs is 3.3 to 3.5 s and the median
about 5.5 s, with pauses of about 36.5 s between blocks (4 pauses in 40-puff sessions, 3 in 32-puff sessions),
consistent with the Methods (3 to 8 s between puffs, 35 s between blocks). Some sessions hold fewer
than 40 puffs (SD178, SD185 and SD186 have 32, SD207's infusion session 14). Some trace values exceed 1,
which a fraction of eye closure cannot (h009 ketamine series up to 2.74, SD186 infusion up to 2.58),
and are tracking artefacts; they are kept, as in the paper's code. Whether the puff frames are the true
puff times cannot be shown from the eye alone; the iEEG coding-dimension notebook (04) shows the puff
times share the LFP clock.

`which_trials` holds 0-based trial indices; in most files it starts at 1, so the first trial is
left out, as the Methods say ("the first trial was excluded").

In [ ]:
print({sub: (len(b["which_trials"]), int(b["which_trials"].min()), int(b["which_trials"].max()))
       for sub, b in beh.items()})

## p50-condition-labels-link: which non-iEEG series is saline

The non-iEEG files do not say which of the two series is the saline visit. The paper's code
(Zenodo 14920570, `human_ieeg/scripts/analyze_airpuff_p50.py`) reads `pre` as saline and `post` as
ketamine (`treatment` saline for `pre_trials`, highdose for `post_trials`), and lists the eight
sessions by participant with their treatment and CADSS. The link holds when the code's participants
are the dataset's, each with both treatments, and the series have the trial counts the paper
reports (40, of which 38 are used). The meaning of `pre` and `post` itself cannot be checked against
anything in the data; the visit order was crossed over (the code's session names put saline first for
h009, h012 and h013 and ketamine first for h011), so `pre` does not mean the first visit.

In [ ]:
subjects, cadss_p50 = he.p50_code_labels()
code_tab = pd.DataFrame([{"session": s, "participant": s.split("_")[0], "treatment": t,
                          "visit": int(s.split("_")[1]), "CADSS": cadss_p50[s]} for s, t in subjects])
print(code_tab.to_string(index=False))
print("participants in the code:", sorted(code_tab.participant.unique()))
print("participants in the dataset:", sorted(p50))
print("each with one saline and one ketamine session:",
      all(sorted(g.treatment) == ["highdose", "saline"] for _, g in code_tab.groupby("participant")))
first = code_tab.sort_values("visit").groupby("participant").first().treatment
print("treatment of the first visit:", first.to_dict())
print("trials per series in the dataset:", {s: (v[1].shape[0], v[2].shape[0]) for s, v in p50.items()})

**Result: the link holds for identity and counts.** The four participants of the code's list (h009,
h011, h012, h013) are the four in the dataset, each with one saline and one ketamine session, and every
series has 40 trials. Which series is saline rests on the code alone: the files do not label it, and
the first visit was saline for three participants and ketamine for one, so the series names do not
encode the visit order either.

## cadss-scores-link: CADSS scores of the code record

The CADSS scores are not in the dataset. The code record holds the iEEG cohort's item scores per
participant and time point (`data/cadss_across_pts.csv`). The link holds when every participant with
behaviour in the dataset is in the table, with a total before ketamine, during infusion and at 110 min.

In [ ]:
cad = he.load_cadss()
print(cad.drop(columns="columns").to_string())
missing = [s for s in he.IEEG_SUBJECTS if s not in cad.index]
print("dataset participants missing from the table:", missing)
print("table participants without behaviour in the dataset:", [p for p in cad.index if p not in he.IEEG_SUBJECTS])
have = cad.loc[[s for s in he.IEEG_SUBJECTS if s in cad.index], ["pre", "infusion", "post"]]
print("dataset participants with all three totals:", int(have.notna().all(axis=1).sum()), "of", len(he.IEEG_SUBJECTS))
print("SD185 columns:", cad.loc["SD185", "columns"])

**Result: the link holds.** All 14 iEEG participants with behaviour in the dataset are in the code
record's CADSS table, each with a total before ketamine, during infusion and at 110 min (the printed
count); the table also holds 7 participants without behaviour in the dataset. The match is by the
participant identifier, which both use. Nothing in the dataset can be compared with the scores.

## fig1e-human-late-eye-closure-ketamine

> In the non-iEEG human cohort, late eye closure normalised by early eye closure is lower during
> ketamine than during saline (0.71 against 0.43, 4 participants).

Method H-EYE (claims file): per participant and series, trials 3 to 40, late (0.3 to 0.8 s) and early
(0.1 to 0.2 s) means; the late/early value is the mean late over the mean early (Open B, reading a).
Paired t-test across the 4 participants. The readings of Open A (first two trials or the first one
excluded) and Open B (ratio of means or mean of ratios) are all shown. Saline is the `pre` series, by
the code's label (link above). The eye-closure traces in the files are already normalised by the
open-eye size; the normalisation (median eye size over the first 2 s of the first trial window,
claims file H-EYE) cannot be repeated, as the eye sizes are not in these files.

In [ ]:
readings = {"A(a) trials 3-40": np.arange(2, 40), "A(b) trials 2-40": np.arange(1, 40)}
p50_rows = []
for sub, (tt, pre, post) in p50.items():
    for rname, idx in readings.items():
        for ratio in ["ratio_of_means", "mean_of_ratios"]:
            for cond, x in [("saline", pre), ("ketamine", post)]:
                e, l, r = he.p50_values(tt, x, idx, ratio)
                p50_rows.append({"subject": sub, "trials": rname, "ratio": ratio, "condition": cond,
                                 "early": e, "late": l, "late/early": r, "n trials": len(idx)})
p50_df = pd.DataFrame(p50_rows)

def paired(df, value, trials, ratio="ratio_of_means"):
    d = df[(df.trials == trials) & (df.ratio == ratio)].pivot(index="subject", columns="condition", values=value)
    t = st.ttest_rel(d.saline, d.ketamine)
    return d, t

out = []
for trials in readings:
    for ratio in ["ratio_of_means", "mean_of_ratios"]:
        d, t = paired(p50_df, "late/early", trials, ratio)
        out.append({"trials": trials, "ratio": ratio, "saline mean": d.saline.mean(), "saline sem": st.sem(d.saline),
                    "ketamine mean": d.ketamine.mean(), "ketamine sem": st.sem(d.ketamine),
                    "t": t.statistic, "p": t.pvalue, "participants lower on ketamine": int((d.ketamine < d.saline).sum())})
print(pd.DataFrame(out).round(4).to_string(index=False))
d, t = paired(p50_df, "late/early", "A(a) trials 3-40")
print("\nPer participant, main reading (A(a), ratio of means):")
print(d.round(3).to_string())
print("expected: saline 0.71 +/- 0.08, ketamine 0.43 +/- 0.12; paper t = 3.703, p = 0.0342")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for a, (sub, (tt, pre, post)) in zip(ax, p50.items()):
    a.plot(tt, pre[2:40].mean(0), label="saline (pre)")
    a.plot(tt, post[2:40].mean(0), label="ketamine (post)")
    a.axvspan(*he.EARLY_W, alpha=0.1); a.axvspan(*he.LATE_W, alpha=0.1, color="r")
    a.set_title(sub); a.set_xlabel("time from puff (s)")
ax[0].set_ylabel("eye closure"); ax[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("plotted range of the mean traces:", round(min(min(v[1][2:40].mean(0).min(), v[2][2:40].mean(0).min()) for v in p50.values()), 3),
      "to", round(max(max(v[1][2:40].mean(0).max(), v[2][2:40].mean(0).max()) for v in p50.values()), 3))

**Result: reproduced.** With trials 3 to 40 and the ratio of trial means, late/early is 0.706 on saline
and 0.426 on ketamine (both within tolerance), lower on ketamine in all 4 participants, paired t = 3.703,
p = 0.0342, the values of Table S3. The other three readings also give ketamine lower with p < 0.05
(table above). The result rests on the code's label of the `pre` series as saline.

## figs1b-human-early-eye-closure-unchanged

> In the non-iEEG cohort, early (reflexive) eye closure does not differ between saline and ketamine
> (0.84 against 0.78).

A claim of absence, judged on the paper's test: paired t-test across the 4 participants of the mean
early eye closure, trials 3 to 40.

In [ ]:
d, t = paired(p50_df, "early", "A(a) trials 3-40")
print(d.round(3).to_string())
print(f"saline {d.saline.mean():.3f} (sem {st.sem(d.saline):.3f}), ketamine {d.ketamine.mean():.3f} (sem {st.sem(d.ketamine):.3f})")
print(f"paired t = {t.statistic:.3f}, p = {t.pvalue:.4f}; difference {d.saline.mean() - d.ketamine.mean():.3f}, "
      f"Cohen dz = {(d.saline - d.ketamine).mean() / (d.saline - d.ketamine).std(ddof=1):.2f}")
d2, t2 = paired(p50_df, "early", "A(b) trials 2-40")
print(f"variant, trials 2-40: saline {d2.saline.mean():.3f}, ketamine {d2.ketamine.mean():.3f}, t = {t2.statistic:.3f}, p = {t2.pvalue:.4f}")
print("expected: saline 0.84 +/- 0.01, ketamine 0.78 +/- 0.04; paper t = 1.622, p = 0.2032")

**Result: reproduced.** Early eye closure is 0.839 on saline and 0.777 on ketamine, paired t = 1.622,
p = 0.2032, not significant, as in Table S3; with trials 2 to 40 the test is also not significant.

## figs1c-human-late-eye-closure-decrease

> In the non-iEEG cohort, late (affective) eye closure alone is lower during ketamine than during
> saline (0.59 against 0.34).

In [ ]:
for trials in readings:
    d, t = paired(p50_df, "late", trials)
    print(f"{trials}: saline {d.saline.mean():.3f} (sem {st.sem(d.saline):.3f}), ketamine {d.ketamine.mean():.3f} "
          f"(sem {st.sem(d.ketamine):.3f}), t = {t.statistic:.3f}, p = {t.pvalue:.4f}, lower in {(d.ketamine < d.saline).sum()} of 4")
print("expected: saline 0.59 +/- 0.07, ketamine 0.34 +/- 0.10; paper t = 3.653, p = 0.0354")

**Result: reproduced.** Late eye closure is 0.591 on saline and 0.343 on ketamine with trials 3 to 40,
paired t = 3.653, p = 0.0354, lower in all 4 participants; trials 2 to 40 give the same conclusion.

## figs1d-human-cadss-saline-ketamine

> In the non-iEEG cohort, dissociation (CADSS) is higher with ketamine than with saline (9.75 against
> 0.50).

The scores are not in the dataset; the criterion allows the code's constants
(`analyze_airpuff_p50.py`, the `cadss` dictionary), read here from the code record.

In [ ]:
cd = code_tab.pivot(index="participant", columns="treatment", values="CADSS")
t = st.ttest_rel(cd.saline, cd.highdose)
print(cd.to_string())
print(f"saline {cd.saline.mean():.2f} (sem {st.sem(cd.saline):.2f}), ketamine {cd.highdose.mean():.2f} (sem {st.sem(cd.highdose):.2f})")
print(f"paired t = {t.statistic:.3f}, p = {t.pvalue:.4f}")
print("expected: saline 0.50 +/- 0.29, ketamine 9.75 +/- 1.25; paper t = 7.034, p = 0.006")

**Result: reproduced, on the code record's scores.** CADSS is 0.50 on saline and 9.75 on ketamine,
paired t = -7.034 (saline minus ketamine), p = 0.0059. These are the code's own numbers; nothing in the
dataset bears on them.

## fig1f-human-subjective-reports

Blocked. The participants' verbal reports are not in the dataset: the four `sub-h*` files hold only
the eye-closure trials and their time axis (listed in the data check), and no other asset holds text
from the non-iEEG cohort.

## figs8a-ieeg-inclusion-per-participant

> Seven iEEG participants met the behavioural inclusion criteria, each showing a significant decrease
> of late/early eye closure from preinfusion to ketamine and a significant recovery from ketamine to
> postinfusion.

Per participant, the per-trial late/early of the trials in `which_trials`, recomputed from the trial
windows (they equal the stored values, data check). Trials are paired across sessions by position.
Paired t-tests preinfusion against ketamine and ketamine against postinfusion, Benjamini-Hochberg over
the two (claims file; code airpuff_utils, Annotator with t-test_paired and fdr_bh). NaN trials (where
the eye was not tracked) are dropped pairwise.

In [ ]:
def inclusion_tests(sub):
    b = beh[sub]
    v = {s: he.per_trial(b, s)["late_over_early"].to_numpy() for s in he.SESSIONS}
    res = {"subject": sub, "n which_trials": len(b["which_trials"])}
    ps = []
    for a, c, lab in [("preinfusion", "infusion", "pre"), ("postinfusion", "infusion", "post")]:
        ok = ~np.isnan(v[a]) & ~np.isnan(v[c])
        t = st.ttest_rel(v[a][ok], v[c][ok])
        res[f"t {lab}-ket"] = t.statistic
        res[f"n {lab}-ket"] = int(ok.sum())
        ps.append(t.pvalue)
    pc = multipletests(ps, method="fdr_bh")[1]
    res["p pre-ket (BH)"], res["p post-ket (BH)"] = pc
    for s in he.SESSIONS:
        res[f"mean {s[:4]}"] = np.nanmean(v[s])
        res[f"sem {s[:4]}"] = st.sem(v[s], nan_policy="omit")
    res["decrease"] = (res["t pre-ket"] > 0) and (pc[0] < 0.05)
    res["recovery"] = (res["t post-ket"] > 0) and (pc[1] < 0.05)
    res["meets both"] = res["decrease"] and res["recovery"]
    return res

inc = pd.DataFrame([inclusion_tests(s) for s in he.IEEG_SUBJECTS]).set_index("subject")
pd.set_option("display.width", 250)
print(inc.loc[he.INCLUDED].round(4).to_string())

In [ ]:
import yaml, pathlib
claims = yaml.safe_load(pathlib.Path("../../claims/001347/10.1126-science.adt3971.yaml").read_text())["claims"]
exp8a = {e["quantity"]: (e["value"], e["tolerance"]) for e in next(c for c in claims if c["id"] == "figs8a-ieeg-inclusion-per-participant")["expected"]}
rows = []
for q, (val, tol) in exp8a.items():
    sub, _, sess = q.split(" ")
    col = {"preinfusion": "mean prei", "ketamine": "mean infu", "postinfusion": "mean post"}[sess]
    m = inc.loc[sub, col]
    rows.append({"quantity": q, "expected": val, "tolerance": tol, "measured": round(m, 3), "within": abs(m - val) <= tol + 1e-9})
cmp8a = pd.DataFrame(rows)
print(cmp8a.to_string(index=False))
print("means within tolerance:", int(cmp8a.within.sum()), "of", len(cmp8a))
print("participants meeting both tests:", int(inc.loc[he.INCLUDED, "meets both"].sum()), "of 7")
print("trial counts (pairs) for the 7:", inc.loc[he.INCLUDED, "n pre-ket"].to_dict())

**Result: reproduced.** All 7 participants show a significant decrease from preinfusion to ketamine and
a significant recovery after it (14 of 14 corrected tests p < 0.05 in the stated direction), with the
trial counts of Table S3 (24, 39, 30, 31, 12, 39 and 39), and 21 of 21 session means within tolerance
(tables above). Thirteen of the 14 t values equal Table S3's to the printed digits; SD245's preinfusion
against ketamine test gives t = 5.288 where the table has 5.2553.

## figs8b-ieeg-excluded-participants

> The other seven iEEG eyepuff participants did not meet the inclusion criteria, each lacking a
> significant ketamine decrease or a significant recovery.

In [ ]:
excl = [s for s in he.IEEG_SUBJECTS if s not in he.INCLUDED]
print(inc.loc[excl, ["n which_trials", "t pre-ket", "p pre-ket (BH)", "t post-ket", "p post-ket (BH)",
                     "mean prei", "mean infu", "mean post", "decrease", "recovery", "meets both"]].round(4).to_string())
print("excluded participants meeting both:", int(inc.loc[excl, "meets both"].sum()), "of", len(excl))
print("participants meeting both, over all 14:", list(inc.index[inc["meets both"]]))

**Result: reproduced.** None of SD163, SD170, SD178, SD184, SD207, SD215 and SD242 meets both criteria,
and the participants that do, over all 14, are the 7 of figs8a (printed list).

## fig2c-ieeg-cohort-selective-affective-decrease

> In the iEEG cohort, ketamine reduces the late (affective) phase of the eyepuff eye closure while the
> early reflexive phase remains.

Fig. 2C is the mean over the 7 participants of each participant's median trace over the trials of
`which_trials` (code, `overlay_sessions_eye_closure_traces_by_pt`). Early and late means are taken on
these median traces. The code plots them on a time axis shifted by 0.05 s (`origin_shift_s`); the
early and late means are shown with the windows unshifted (the stored windows) and shifted 0.05 s
later. The paired t-test of early eye closure, ketamine against preinfusion, is the criterion's.

In [ ]:
rows = []
traces = {}
for sub in he.INCLUDED:
    b = beh[sub]
    for sess in he.SESSIONS:
        med = np.nanmedian(b[sess]["trials"][b["which_trials"]], axis=0)[None, :]
        traces[(sub, sess)] = med[0]
        for shift in [0.0, 0.05]:
            rows.append({"subject": sub, "session": sess, "shift": shift,
                         "early": he.window_mean(med, he.EARLY_W, shift)[0],
                         "late": he.window_mean(med, he.LATE_W, shift)[0]})
c2 = pd.DataFrame(rows)
for shift in [0.0, 0.05]:
    d = c2[c2["shift"] == shift]
    m = d.groupby("session")[["early", "late"]].agg(["mean", "sem"]).loc[he.SESSIONS]
    print(f"window shift {shift} s, mean (sem) over 7 participants:")
    print(m.round(3).to_string())
    e = d.pivot(index="subject", columns="session", values="early")
    l = d.pivot(index="subject", columns="session", values="late")
    te = st.ttest_rel(e.preinfusion, e.infusion)
    print(f"  early, pre against ketamine: paired t = {te.statistic:.3f}, p = {te.pvalue:.4f} (positive t = ketamine lower)")
    for other in ["preinfusion", "postinfusion"]:
        tl = st.ttest_rel(l[other], l.infusion)
        print(f"  late, {other} against ketamine: t = {tl.statistic:.3f}, p = {tl.pvalue:.4f}; ketamine lower in {(l.infusion < l[other]).sum()} of 7")
    print()
print("per participant (no shift):")
print(c2[c2["shift"] == 0].pivot(index="subject", columns="session", values=["early", "late"]).round(3).to_string())

In [ ]:
tt = (np.arange(800) - 400) / he.FPS
fig, ax = plt.subplots(figsize=(5, 3))
allv = []
for sess in he.SESSIONS:
    m = np.mean([traces[(s, sess)] for s in he.INCLUDED], axis=0)
    allv.append(m)
    ax.plot(tt, m, label=sess)
ax.set_xlim(-0.5, 1.5); ax.set_xlabel("time from puff (s)"); ax.set_ylabel("eye closure"); ax.legend()
plt.show()
sel = (tt >= -0.5) & (tt <= 1.5)
print("plotted range:", round(min(a[sel].min() for a in allv), 3), "to", round(max(a[sel].max() for a in allv), 3))

**Result: reproduced.** Across the 7 participants, mean late eye closure of the median traces is lower
on ketamine than at preinfusion and at postinfusion, and early eye closure on ketamine is not
significantly lower than at preinfusion by the paired t-test, with the windows unshifted and shifted
by 0.05 s (numbers above).

## fig2d-ieeg-cadss-reversible

> In the 7 included iEEG participants, dissociation (CADSS) rises during ketamine and returns to
> baseline after infusion (0.86, 22.43, 0.86).

Scores from the code record (link above); infusion is the maximum over the time points up to 50 min,
with SD185's retrospective rating in place of its infusion rating (claims file; code
`mean_cadss_by_sess`). Paired t-tests with Benjamini-Hochberg over the two.

In [ ]:
c7 = cad.loc[he.INCLUDED, ["pre", "infusion", "post", "retro"]].copy()
c7.loc["SD185", "infusion"] = c7.loc["SD185", "retro"]
print(c7.to_string())
t1 = st.ttest_rel(c7.pre, c7.infusion); t2 = st.ttest_rel(c7.post, c7.infusion)
pc = multipletests([t1.pvalue, t2.pvalue], method="fdr_bh")[1]
for col in ["pre", "infusion", "post"]:
    print(f"{col}: mean {c7[col].mean():.2f}, sem {st.sem(c7[col]):.2f}")
print(f"pre against ketamine t = {t1.statistic:.3f}, corrected p = {pc[0]:.4f}; post against ketamine t = {t2.statistic:.3f}, corrected p = {pc[1]:.4f}")
c7b = cad.loc[he.INCLUDED, ["pre", "infusion", "post"]]
print(f"variant without the retrospective rating for SD185: infusion mean {c7b.infusion.mean():.2f}")
print("expected: 0.86 +/- 0.46, 22.43 +/- 4.75, 0.86 +/- 0.46; paper t = 4.807 and 4.512, p = 0.004 and 0.004")

**Result: reproduced, on the code record's scores.** CADSS is 0.86 before, 22.43 during and 0.86 after
ketamine, with both corrected p = 0.004; these are the code record's numbers and nothing in the dataset
bears on them.

## figs14g-cadss-correlates-with-affective-decrease

> Across participants, the ketamine increase in CADSS correlates with the ketamine reduction of
> normalised affective eye closure (Spearman 0.744, 10 participants).

Participants: the iEEG eyepuff participants with behaviour and CADSS (the dataset has no bolus
participant), excluding those whose median per-trial late/early at preinfusion is below 0.5 (claims file;
code `load_cadss_and_behavior`). CADSS change: infusion minus preinfusion total, SD185's retrospective
rating as its infusion value. Eye-closure change, reading (a), the code's: mean of the preinfusion and
postinfusion medians minus the infusion median; reading (b): preinfusion median minus infusion median.

In [ ]:
rows = []
for sub in he.IEEG_SUBJECTS:
    b = beh[sub]
    med = {s: np.nanmedian(he.per_trial(b, s)["late_over_early"]) for s in he.SESSIONS}
    inf = cad.loc[sub, "retro"] if sub == "SD185" else cad.loc[sub, "infusion"]
    rows.append({"subject": sub, "median pre": med["preinfusion"], "median ket": med["infusion"],
                 "median post": med["postinfusion"],
                 "change (a)": (med["preinfusion"] + med["postinfusion"]) / 2 - med["infusion"],
                 "change (b)": med["preinfusion"] - med["infusion"],
                 "CADSS change": inf - cad.loc[sub, "pre"]})
g = pd.DataFrame(rows).set_index("subject")
g["kept"] = g["median pre"] >= 0.5
print(g.round(3).to_string())
k = g[g.kept]
print("participants kept:", len(k))
for r in ["change (a)", "change (b)"]:
    rho = st.spearmanr(k["CADSS change"], k[r])
    print(f"reading {r}: Spearman rho = {rho.statistic:.3f}, p = {rho.pvalue:.4f}")
rho_all = st.spearmanr(g["CADSS change"], g["change (a)"])
print(f"variant, all 14 participants, reading (a): rho = {rho_all.statistic:.3f}, p = {rho_all.pvalue:.4f}")
print("expected: rho 0.744 +/- 0.074, p = 0.014, 10 participants")

**Result: reproduced under reading (a).** Ten participants remain after the exclusion (SD170, SD184,
SD211 and SD215 have preinfusion medians below 0.5), and with the code's eye-closure change the
Spearman correlation with the CADSS change is 0.744, p = 0.0136, the paper's value. Under reading (b),
ketamine minus preinfusion, rho is 0.561 and p = 0.0916, not significant. The CADSS values are the code
record's.

## Summary

| Claim | Outcome |
|---|---|
| p50-condition-labels-link | reproduced (identity and counts; the saline label rests on the code) |
| cadss-scores-link | reproduced |
| fig1e-human-late-eye-closure-ketamine | reproduced |
| figs1b-human-early-eye-closure-unchanged | reproduced |
| figs1c-human-late-eye-closure-decrease | reproduced |
| figs1d-human-cadss-saline-ketamine | reproduced (code record's scores) |
| fig1f-human-subjective-reports | blocked, data missing |
| figs8a-ieeg-inclusion-per-participant | reproduced |
| figs8b-ieeg-excluded-participants | reproduced |
| fig2c-ieeg-cohort-selective-affective-decrease | reproduced |
| fig2d-ieeg-cadss-reversible | reproduced (code record's scores) |
| figs14g-cadss-correlates-with-affective-decrease | reproduced under reading (a), the code's (code record's scores) |

Differences from the paper's method: the eye-closure traces are taken as deposited (already normalised
to the open eye); the eye sizes and the DeepLabCut tracking are not in these files. The saline label of
the non-iEEG series and all CADSS scores come from the code record.

About the dataset: the non-iEEG files do not say which series is saline; the CADSS scores and the
verbal reports are not deposited; the iEEG behaviour files are internally consistent (trial windows,
stored per-trial values and their ratio agree with the continuous trace).